In [149]:
import common
from db import get_engine
engine = get_engine()

In [150]:
df = pd.read_sql_table(
    "flights_total",
    con=engine,
    schema="bronze"
)

df.head()

,MONTH,DAY,DAY_OF_WEEK,AIRLINE,ORIGIN_AIRPORT,DESTINATION_AIRPORT,DISTANCE,SCHEDULED_DEPARTURE,SCHEDULED_ARRIVAL,DEPARTURE_DELAY,...,FLIGHT_WEATHER_TIME,WEATHER_TEMPERATURE,WEATHER_PRECIPITATION,WEATHER_WIND_SPEED,WEATHER_VISIBILITY,WEATHER_CONDITION,WEATHER_SURFACE_PRESSURE,WEATHER_WIND_DIRECTION,WEATHER_SNOWFALL,WEATHER_WIND_GUSTS
0,11,5,4,WN,TPA,MDW,997,1815,2000,20.0,...,2015-11-05 18:00:00,30.7,0.0,15.1,NaN,0.0,1018.9,87.0,0.0,29.2
1,8,11,2,UA,SFO,EWR,2565,2040,509,4.0,...,2015-08-11 20:00:00,21.7,0.0,10.2,NaN,0.0,1013.4,278.0,0.0,29.2
2,3,31,2,WN,MDW,STL,251,840,950,9.0,...,2015-03-31 08:00:00,11.3,0.0,28.0,NaN,0.0,984.1,257.0,0.0,51.5
3,6,23,2,B6,SFO,JFK,2586,1335,2220,16.0,...,2015-06-23 13:00:00,12.9,0.0,5.7,NaN,3.0,1016.3,235.0,0.0,18.4
4,11,29,7,WN,LAS,DAL,1067,1820,2250,29.0,...,2015-11-29 18:00:00,6.6,0.0,4.1,NaN,0.0,947.1,15.0,0.0,14.8


# Flights Data Cleaning

## 1.1. Sütunların təsviri

Bu məlumat dəsti uçuşların xüsusiyyətlərini, gecikmə göstəricilərini və uçuşlarla əlaqəli hava şəraiti məlumatlarını əhatə edir. Sütunlar uçuş gecikmələrinin təhlili və gecikmələrin proqnozlaşdırılması məqsədilə istifadə olunur.

### A. Uçuş tarixi və marşrut məlumatları

| Sütun | İzahı |
|---|---|
| `MONTH` | Uçuşun həyata keçirildiyi ay |
| `DAY` | Uçuşun həyata keçirildiyi gün |
| `DAY_OF_WEEK` | Həftənin günü |
| `AIRLINE` | Aviaşirkətin qısa kodu |
| `ORIGIN_AIRPORT` | Uçuşun başladığı hava limanının kodu |
| `DESTINATION_AIRPORT` | Uçuşun təyin olunmuş son hava limanının kodu |
| `DISTANCE` | Başlanğıc və təyinat hava limanları arasındakı məsafə |

### B. Planlaşdırılmış vaxtlar və gecikmələr

| Sütun | İzahı |
|---|---|
| `SCHEDULED_DEPARTURE` | Planlaşdırılmış uçuş vaxtı, HHMM formatında |
| `SCHEDULED_ARRIVAL` | Planlaşdırılmış eniş vaxtı, HHMM formatında |
| `DEPARTURE_DELAY` | Uçuşun planlaşdırılmış vaxtdan neçə dəqiqə gecikdiyi; mənfi dəyər erkən uçuşu göstərə bilər |
| `ARRIVAL_DELAY` | Faktiki enişin planlaşdırılmış vaxtdan neçə dəqiqə gecikdiyi; mənfi dəyər erkən enişi göstərə bilər |

### C. Gecikmə səbəbləri

| Sütun | İzahı |
|---|---|
| `AIR_SYSTEM_DELAY` | Hava hərəkətinin idarə edilməsi sistemi ilə əlaqəli gecikmə |
| `SECURITY_DELAY` | Təhlükəsizlik məsələləri ilə əlaqəli gecikmə |
| `AIRLINE_DELAY` | Aviaşirkətin fəaliyyəti ilə əlaqəli gecikmə |
| `LATE_AIRCRAFT_DELAY` | Əvvəlki uçuşdan gec gələn təyyarə səbəbindən yaranan gecikmə |
| `WEATHER_DELAY` | Hava şəraiti səbəbindən yaranan gecikmə |

### D. Uçuşun statusu və müddət göstəriciləri

| Sütun | İzahı |
|---|---|
| `CANCELLED` | Uçuşun ləğv edilib-edilmədiyini göstərir: 1 — bəli, 0 — xeyr |
| `DIVERTED` | Uçuşun başqa hava limanına yönləndirilib-yönləndirilmədiyini göstərir: 1 — bəli, 0 — xeyr |
| `TAXI_OUT` | Təyyarənin dayanacaqdan uçuş zolağına qədər hərəkət müddəti, dəqiqə ilə |
| `TAXI_IN` | Təyyarənin enişdən sonra dayanacaq yerinə qədər hərəkət müddəti, dəqiqə ilə |
| `AIR_TIME` | Təyyarənin havada olduğu müddət, dəqiqə ilə |

### E. Aviaşirkət və hava limanı haqqında əlavə məlumatlar

| Sütun | İzahı |
|---|---|
| `AIRLINE_NAME` | Aviaşirkətin tam adı |
| `ORIGIN_AIRPORT_NAME` | Başlanğıc hava limanının adı |
| `ORIGIN_CITY` | Başlanğıc hava limanının yerləşdiyi şəhər |
| `ORIGIN_STATE` | Başlanğıc hava limanının yerləşdiyi ştat |
| `ORIGIN_COUNTRY` | Başlanğıc hava limanının yerləşdiyi ölkə |
| `DESTINATION_AIRPORT_NAME` | Təyinat hava limanının adı |
| `DESTINATION_CITY` | Təyinat hava limanının yerləşdiyi şəhər |
| `DESTINATION_STATE` | Təyinat hava limanının yerləşdiyi ştat |
| `DESTINATION_COUNTRY` | Təyinat hava limanının yerləşdiyi ölkə |

### F. Hava məlumatları ilə əlaqələndirmə

| Sütun | İzahı |
|---|---|
| `FLIGHT_WEATHER_TIME` | Uçuş qeydi ilə hava məlumatlarını əlaqələndirmək üçün istifadə olunan tarix-saat göstəricisi |
| `WEATHER_TEMPERATURE` | Havanın temperaturu |
| `WEATHER_PRECIPITATION` | Yağıntının miqdarı |
| `WEATHER_WIND_SPEED` | Küləyin sürəti |
| `WEATHER_VISIBILITY` | Görünüş məsafəsi |
| `WEATHER_CONDITION` | Hava şəraitini təsvir edən göstərici |
| `WEATHER_SURFACE_PRESSURE` | Yer səthində atmosfer təzyiqi |
| `WEATHER_WIND_DIRECTION` | Küləyin istiqaməti

### 1.2. Datanın ilkin vəziyyətinin yoxlanılması

In [151]:
df.shape
print("Sətir və sütun sayı:", df.shape)

Sətir və sütun sayı: (999657, 40)


In [152]:
df.head()

,MONTH,DAY,DAY_OF_WEEK,AIRLINE,ORIGIN_AIRPORT,DESTINATION_AIRPORT,DISTANCE,SCHEDULED_DEPARTURE,SCHEDULED_ARRIVAL,DEPARTURE_DELAY,...,FLIGHT_WEATHER_TIME,WEATHER_TEMPERATURE,WEATHER_PRECIPITATION,WEATHER_WIND_SPEED,WEATHER_VISIBILITY,WEATHER_CONDITION,WEATHER_SURFACE_PRESSURE,WEATHER_WIND_DIRECTION,WEATHER_SNOWFALL,WEATHER_WIND_GUSTS
0,11,5,4,WN,TPA,MDW,997,1815,2000,20.0,...,2015-11-05 18:00:00,30.7,0.0,15.1,NaN,0.0,1018.9,87.0,0.0,29.2
1,8,11,2,UA,SFO,EWR,2565,2040,509,4.0,...,2015-08-11 20:00:00,21.7,0.0,10.2,NaN,0.0,1013.4,278.0,0.0,29.2
2,3,31,2,WN,MDW,STL,251,840,950,9.0,...,2015-03-31 08:00:00,11.3,0.0,28.0,NaN,0.0,984.1,257.0,0.0,51.5
3,6,23,2,B6,SFO,JFK,2586,1335,2220,16.0,...,2015-06-23 13:00:00,12.9,0.0,5.7,NaN,3.0,1016.3,235.0,0.0,18.4
4,11,29,7,WN,LAS,DAL,1067,1820,2250,29.0,...,2015-11-29 18:00:00,6.6,0.0,4.1,NaN,0.0,947.1,15.0,0.0,14.8


In [153]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 999657 entries, 0 to 999656
Data columns (total 40 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   MONTH                     999657 non-null  int64         
 1   DAY                       999657 non-null  int64         
 2   DAY_OF_WEEK               999657 non-null  int64         
 3   AIRLINE                   999657 non-null  str           
 4   ORIGIN_AIRPORT            999657 non-null  str           
 5   DESTINATION_AIRPORT       999657 non-null  str           
 6   DISTANCE                  999657 non-null  int64         
 7   SCHEDULED_DEPARTURE       999657 non-null  int64         
 8   SCHEDULED_ARRIVAL         999657 non-null  int64         
 9   DEPARTURE_DELAY           984790 non-null  float64       
 10  ARRIVAL_DELAY             981515 non-null  float64       
 11  AIR_SYSTEM_DELAY          182526 non-null  float64       
 12  SECURITY_DELA

## 2. Data tiplərinin yoxlanılması və düzəldilməsi

Bu mərhələdə sütunların data tipləri təhlil edilir və onların məzmununa uyğunlaşdırılır. Məqsəd yaddaş istifadəsini optimallaşdırmaq, məlumatların düzgünlüyünü qorumaq və sonrakı analiz mərhələlərinə hazırlıq görməkdir.

İNT tipli sütunların yaddaş həcminin optimallaşdırılması

In [154]:
# Məsafə sütunu
df["DISTANCE"] = df["DISTANCE"].astype("int32")

In [155]:
# Kiçik diapazonlu tam ədəd sütunları
small_int_cols = [
    "MONTH",
    "DAY",
    "DAY_OF_WEEK",
    "CANCELLED",
    "DIVERTED"]

for col in small_int_cols:
    df[col] = df[col].astype("int8")

In [156]:
df[["SCHEDULED_DEPARTURE", "SCHEDULED_ARRIVAL"]].agg(
    ["min", "max"]
)

,SCHEDULED_DEPARTURE,SCHEDULED_ARRIVAL
min,1,1
max,2359,2359


In [157]:
# Planlaşdırılmış vaxtlar
df["SCHEDULED_DEPARTURE"] = df["SCHEDULED_DEPARTURE"].astype("int16")
df["SCHEDULED_ARRIVAL"] = df["SCHEDULED_ARRIVAL"].astype("int16")

Float tipli sütunlarının yaddaş həcminin optimallaşdırılması

In [158]:
float_cols = [
    "DEPARTURE_DELAY",
    "ARRIVAL_DELAY",
    "AIR_SYSTEM_DELAY",
    "SECURITY_DELAY",
    "AIRLINE_DELAY",
    "LATE_AIRCRAFT_DELAY",
    "WEATHER_DELAY",
    "TAXI_OUT",
    "TAXI_IN",
    "AIR_TIME"
]

df[float_cols] = df[float_cols].astype("float32")

In [159]:
df[float_cols].dtypes

DEPARTURE_DELAY        float32
ARRIVAL_DELAY          float32
AIR_SYSTEM_DELAY       float32
SECURITY_DELAY         float32
AIRLINE_DELAY          float32
LATE_AIRCRAFT_DELAY    float32
WEATHER_DELAY          float32
TAXI_OUT               float32
TAXI_IN                float32
AIR_TIME               float32
dtype: object

Hava göstəricilərinin data tiplərinin düzəldilməsi

In [160]:
weather_cols = [
    "WEATHER_TEMPERATURE",
    "WEATHER_PRECIPITATION",
    "WEATHER_WIND_SPEED",
    "WEATHER_CONDITION",
    "WEATHER_SURFACE_PRESSURE",
    "WEATHER_WIND_DIRECTION",
    "WEATHER_SNOWFALL",
    "WEATHER_WIND_GUSTS"]

df[weather_cols] = df[weather_cols].astype("float32")

## 3.Lazım olmayan sütunların silinməsi

In [161]:
df["WEATHER_VISIBILITY"].isna().sum(), len(df)

(np.int64(999657), 999657)

Burada görünür ki, "Weather_Visibility" sütunu ümumilikdə boşdur. Buna görə onu cədvəldən silə bilərik.

In [162]:
df = df.drop(columns=["WEATHER_VISIBILITY"])

## 4.Təkrarlanan sətirlərin yoxlanılması

In [163]:
duplicate_count = df.duplicated().sum()

print("Təkrarlanan sətirlərin sayı:", duplicate_count)
print("Duplikatların payı (%):", round(duplicate_count / len(df) * 100, 2))

Təkrarlanan sətirlərin sayı: 0
Duplikatların payı (%): 0.0


Datada duplikatlara rast gəlinmədi.

## 5. Boş dəyərlərin yoxlanılması

In [164]:
df.isna().sum()[df.isna().sum() > 0]

DEPARTURE_DELAY              14867
ARRIVAL_DELAY                18142
AIR_SYSTEM_DELAY            817131
SECURITY_DELAY              817131
AIRLINE_DELAY               817131
LATE_AIRCRAFT_DELAY         817131
WEATHER_DELAY               817131
TAXI_OUT                     15375
TAXI_IN                      15991
AIR_TIME                     18142
WEATHER_TEMPERATURE         226557
WEATHER_PRECIPITATION       226557
WEATHER_WIND_SPEED          226557
WEATHER_CONDITION           226557
WEATHER_SURFACE_PRESSURE    226557
WEATHER_WIND_DIRECTION      226557
WEATHER_SNOWFALL            226557
WEATHER_WIND_GUSTS          226557
dtype: int64

In [165]:
df[["CANCELLED", "DIVERTED"]].value_counts()

CANCELLED  DIVERTED
0          0           981515
1          0            15527
0          1             2615
Name: count, dtype: int64

In [166]:
df.loc[
    df["ARRIVAL_DELAY"].isna(),
    ["CANCELLED", "DIVERTED"]
].value_counts()

CANCELLED  DIVERTED
1          0           15527
0          1            2615
Name: count, dtype: int64

In [167]:
df.loc[
    df["DEPARTURE_DELAY"].isna(),
    ["CANCELLED", "DIVERTED"]
].value_counts()

CANCELLED  DIVERTED
1          0           14867
Name: count, dtype: int64

Arrival_delay və Departure_delay sütunlarındakı boş dəyərlərin hamısı ləğv edilmiş və dəyişdirilmiş reyslərə aiddir. Buna görə də onları olduğu kimi saxlayırıq. Növbəti addımda gecikmə səbəbləri sütunarını yoxlayaq.

In [168]:
df.loc[
    df["AIR_SYSTEM_DELAY"].isna(),
    ["CANCELLED", "DIVERTED"]].value_counts()

CANCELLED  DIVERTED
0          0           798989
1          0            15527
0          1             2615
Name: count, dtype: int64

Nəticədə gördük ki, AIR_SYSTEM_DELAY boş olan uçuşların:
- 798 989-u ləğv edilməyib və istiqaməti dəyişdirilməyib.
- 15 527-si ləğv edilib.
- 2 615-i istiqaməti dəyişdirilib.
Bu göstərici bütün gecikmə səbəbləri sütunlarında eynidir.

In [169]:
df.loc[
    df["AIR_SYSTEM_DELAY"].isna() & (df["ARRIVAL_DELAY"] > 0),
    ["ARRIVAL_DELAY", "AIR_SYSTEM_DELAY", "SECURITY_DELAY",
     "AIRLINE_DELAY", "LATE_AIRCRAFT_DELAY", "WEATHER_DELAY"]
].shape

(175944, 6)

Boş olan sətirlərdən 175 944-də ARRIVAL_DELAY > 0-dır. Deməli, gecikmə səbəbi boşdursa, bu, mütləq gecikmə olmadığı demək deyil.

In [170]:
df[[
    "AIR_SYSTEM_DELAY",
    "SECURITY_DELAY",
    "AIRLINE_DELAY",
    "LATE_AIRCRAFT_DELAY",
    "WEATHER_DELAY"
]].notna().sum()

AIR_SYSTEM_DELAY       182526
SECURITY_DELAY         182526
AIRLINE_DELAY          182526
LATE_AIRCRAFT_DELAY    182526
WEATHER_DELAY          182526
dtype: int64

Bu sütunlardakı boş dəyərləri hələ 0 ilə doldurmuruq. Çünki gecikmənin səbəbi məlum olmaya bilər.

In [171]:
weather_cols = [
    "WEATHER_TEMPERATURE",
    "WEATHER_PRECIPITATION",
    "WEATHER_WIND_SPEED",
    "WEATHER_CONDITION",
    "WEATHER_SURFACE_PRESSURE",
    "WEATHER_WIND_DIRECTION",
    "WEATHER_SNOWFALL",
    "WEATHER_WIND_GUSTS"
]

df[weather_cols].isna().sum()

WEATHER_TEMPERATURE         226557
WEATHER_PRECIPITATION       226557
WEATHER_WIND_SPEED          226557
WEATHER_CONDITION           226557
WEATHER_SURFACE_PRESSURE    226557
WEATHER_WIND_DIRECTION      226557
WEATHER_SNOWFALL            226557
WEATHER_WIND_GUSTS          226557
dtype: int64

In [172]:
df.loc[
    df[weather_cols].isna().all(axis=1),
    ["CANCELLED", "DIVERTED"]
].value_counts()


CANCELLED  DIVERTED
0          0           222134
1          0             3837
0          1              586
Name: count, dtype: int64

222134 — ləğv edilməyib, istiqaməti dəyişdirilməyib



3837 — ləğv edilib



586 — istiqaməti dəyişdirilib

In [173]:
df.loc[
    df[weather_cols].isna().all(axis=1),
    "FLIGHT_WEATHER_TIME"
].isna().sum()

np.int64(0)

Boş olan sətirlərdə saat məlumatları olduğuna görə onları hava datasından bərpa edə bilərik.

In [174]:
df[["TAXI_OUT", "TAXI_IN", "AIR_TIME"]].isna().sum()

TAXI_OUT    15375
TAXI_IN     15991
AIR_TIME    18142
dtype: int64

In [175]:
cols = ["TAXI_OUT", "TAXI_IN", "AIR_TIME"]

for col in cols:
    print(f"\n{col}")
    print(
        df.loc[df[col].isna(), ["CANCELLED", "DIVERTED"]]
        .value_counts()
    )


TAXI_OUT
CANCELLED  DIVERTED
1          0           15375
Name: count, dtype: int64

TAXI_IN
CANCELLED  DIVERTED
1          0           15527
0          1             464
Name: count, dtype: int64

AIR_TIME
CANCELLED  DIVERTED
1          0           15527
0          1            2615
Name: count, dtype: int64


Hər 3 sütundakı boş dəyərlər cancelled və diverted reyslər ilə bağlıdır.Buna görə onları 0 ilə əvəz etmirik. Çünki boş dəyər ölçmənin mövcud olmadığını göstərir. Onu sıfıra çevirmək məlumatın mənasını dəyişə bilər.

In [176]:
missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)

display(missing)

SECURITY_DELAY              817131
AIR_SYSTEM_DELAY            817131
WEATHER_DELAY               817131
LATE_AIRCRAFT_DELAY         817131
AIRLINE_DELAY               817131
WEATHER_WIND_DIRECTION      226557
WEATHER_SURFACE_PRESSURE    226557
WEATHER_WIND_SPEED          226557
WEATHER_PRECIPITATION       226557
WEATHER_TEMPERATURE         226557
WEATHER_WIND_GUSTS          226557
WEATHER_CONDITION           226557
WEATHER_SNOWFALL            226557
ARRIVAL_DELAY                18142
AIR_TIME                     18142
TAXI_IN                      15991
TAXI_OUT                     15375
DEPARTURE_DELAY              14867
dtype: int64

## 5. Zaman sütunlarının optimallaşdırılması

In [177]:
print(df[[
    "MONTH",
    "DAY",
    "SCHEDULED_DEPARTURE",
    "FLIGHT_WEATHER_TIME"
]].head(10))

print("\nData tipləri:")
print(df[[
    "MONTH",
    "DAY",
    "SCHEDULED_DEPARTURE",
    "FLIGHT_WEATHER_TIME"
]].dtypes)

   MONTH  DAY  SCHEDULED_DEPARTURE FLIGHT_WEATHER_TIME
0     11    5                 1815 2015-11-05 18:00:00
1      8   11                 2040 2015-08-11 20:00:00
2      3   31                  840 2015-03-31 08:00:00
3      6   23                 1335 2015-06-23 13:00:00
4     11   29                 1820 2015-11-29 18:00:00
5     12   25                 1020 2015-12-25 10:00:00
6     10    1                 1030 2015-10-01 10:00:00
7      7   10                  600 2015-07-10 06:00:00
8      7   18                  514 2015-07-18 05:00:00
9     11    9                  940 2015-11-09 09:00:00

Data tipləri:
MONTH                            int8
DAY                              int8
SCHEDULED_DEPARTURE             int16
FLIGHT_WEATHER_TIME    datetime64[us]
dtype: object


In [178]:
df["YEAR"] = 2015

In [179]:
print(df[["YEAR", "MONTH", "DAY", "FLIGHT_WEATHER_TIME"]].head())
print(df["YEAR"].dtype)

   YEAR  MONTH  DAY FLIGHT_WEATHER_TIME
0  2015     11    5 2015-11-05 18:00:00
1  2015      8   11 2015-08-11 20:00:00
2  2015      3   31 2015-03-31 08:00:00
3  2015      6   23 2015-06-23 13:00:00
4  2015     11   29 2015-11-29 18:00:00
int64


HHMM tipli sütunları saat formatına salaq.

In [180]:
df[["SCHEDULED_DEPARTURE", "SCHEDULED_ARRIVAL"]].head(10)

,SCHEDULED_DEPARTURE,SCHEDULED_ARRIVAL
0,1815,2000
1,2040,509
2,840,950
3,1335,2220
4,1820,2250
5,1020,1230
6,1030,1357
7,600,935
8,514,930
9,940,1536


In [181]:
for col in ["SCHEDULED_DEPARTURE", "SCHEDULED_ARRIVAL"]:
    df[col] = (
        df[col]
        .astype("Int64")
        .astype(str)
        .str.zfill(4)
        .str[:2]
        + ":"
        + df[col].astype("Int64").astype(str).str.zfill(4).str[2:]
    )

In [182]:
df[["SCHEDULED_DEPARTURE", "SCHEDULED_ARRIVAL"]].head(10)

,SCHEDULED_DEPARTURE,SCHEDULED_ARRIVAL
0,18:15,20:00
1,20:40,05:09
2,08:40,09:50
3,13:35,22:20
4,18:20,22:50
5,10:20,12:30
6,10:30,13:57
7,06:00,09:35
8,05:14,09:30
9,09:40,15:36


In [183]:
for col in ["SCHEDULED_DEPARTURE", "SCHEDULED_ARRIVAL"]:
    print(f"\n{col}")
    print("Tip:", df[col].dtype)
    print(df[col].dropna().astype(str).unique()[:15])


SCHEDULED_DEPARTURE
Tip: str
<ArrowStringArray>
['18:15', '20:40', '08:40', '13:35', '18:20', '10:20', '10:30', '06:00',
 '05:14', '09:40', '19:35', '06:05', '17:15', '16:55', '16:30']
Length: 15, dtype: str

SCHEDULED_ARRIVAL
Tip: str
<ArrowStringArray>
['20:00', '05:09', '09:50', '22:20', '22:50', '12:30', '13:57', '09:35',
 '09:30', '15:36', '21:05', '07:25', '19:39', '16:48', '17:30']
Length: 15, dtype: str


In [184]:
from datetime import time
import pandas as pd

for col in ["SCHEDULED_DEPARTURE", "SCHEDULED_ARRIVAL"]:
    df[col] = pd.to_datetime(
        df[col],
        format="%H:%M",
        errors="coerce"
    ).apply(lambda x: x.time() if pd.notna(x) else None)

In [185]:
df[["SCHEDULED_DEPARTURE", "SCHEDULED_ARRIVAL"]].dtypes

SCHEDULED_DEPARTURE    object
SCHEDULED_ARRIVAL      object
dtype: object

## 6. Yeni sütunların yaradılması

In [192]:
df["IS_DELAYED"] = pd.Series(
    pd.NA, index=df.index, dtype="Int8"
)

df.loc[df["ARRIVAL_DELAY"] >= 15, "IS_DELAYED"] = 1
df.loc[df["ARRIVAL_DELAY"] < 15, "IS_DELAYED"] = 0

df["IS_DELAYED"].value_counts(dropna=False)

IS_DELAYED
0       798989
1       182526
<NA>     18142
Name: count, dtype: Int64

In [193]:
df["MONTH"] = df["MONTH"].map({
    1: "January",
    2: "February",
    3: "March",
    4: "April",
    5: "May",
    6: "June",
    7: "July",
    8: "August",
    9: "September",
    10: "October",
    11: "November",
    12: "December"
})

In [194]:
df["MONTH"].value_counts()

MONTH
July         89974
August       87623
March        86649
June         86385
May          85504
April        83792
December     82536
October      82131
January      80869
November     80361
September    79792
February     74041
Name: count, dtype: int64

In [195]:
df["FLIGHT_STATUS"] = "Completed"

df.loc[df["CANCELLED"] == 1, "FLIGHT_STATUS"] = "Cancelled"
df.loc[df["DIVERTED"] == 1, "FLIGHT_STATUS"] = "Diverted"

In [196]:
df["FLIGHT_STATUS"].value_counts()

FLIGHT_STATUS
Completed    981515
Cancelled     15527
Diverted       2615
Name: count, dtype: int64

In [197]:
df["IS_WEEKEND"] = df["DAY_OF_WEEK"].isin([6, 7]).astype("int8")

1 — həftəsonu



0 — həftəiçi

In [198]:
df["IS_WEEKEND"].value_counts()

IS_WEEKEND
0    738808
1    260849
Name: count, dtype: int64

In [199]:
df["DEPARTURE_PERIOD"] = df["SCHEDULED_DEPARTURE"].apply(
    lambda x: "Night" if x.hour < 6 else
              "Morning" if x.hour < 12 else
              "Afternoon" if x.hour < 18 else
              "Evening"
)

In [200]:
df["DEPARTURE_PERIOD"].value_counts()

DEPARTURE_PERIOD
Morning      388979
Afternoon    368276
Evening      218254
Night         24148
Name: count, dtype: int64

In [201]:
df["IS_LONG_DISTANCE"] = (df["DISTANCE"] >= 1500).astype("int8")

In [202]:
df["IS_LONG_DISTANCE"].value_counts()

IS_LONG_DISTANCE
0    864207
1    135450
Name: count, dtype: int64

## 7. Təmizlənmiş datanın sql database yazıması

In [206]:
df.to_sql(
    name="flights_clean",
    con=engine,
    schema="silver",
    if_exists="replace",
    index=False,
    chunksize=1000
)

33979

In [207]:
pd.read_sql(
    "SELECT COUNT(*) AS total_rows FROM silver.flights_clean",
    con=engine
)

,total_rows
0,999657


In [208]:
print("Pandas:", len(df))

pd.read_sql(
    "SELECT COUNT(*) AS total_rows FROM silver.flights_clean",
    con=engine
)

Pandas: 999657


,total_rows
0,999657


In [209]:
pd.read_sql("""
SELECT TABLE_NAME
FROM INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA = 'silver'
ORDER BY TABLE_NAME
""", con=engine)

,TABLE_NAME
0,flights_clean
